# Memory and Sharding, FSDP and DeepSpeed ZeRO

## TLDR

In notebook 01 every GPU held a full copy of the model. That works for gpt2 but
not for a large foundation model, where parameters, gradients, and optimizer
states overflow a single GPU. The fix is sharding. Split those tensors across
the data-parallel GPUs and gather each piece only when needed. You do it two
ways here, PyTorch FSDP and DeepSpeed ZeRO, both launched by the same Ray Train
`TorchTrainer` you already know. You will watch each rank hold only its slice of
the model.


## Introduction

Recall the memory math from notebook 00. A 1 billion parameter model trained
with Adam needs roughly 28 GB, four times the size of the weights, because of
gradients, two optimizer moments, and activations. Plain data parallel, the DDP
you used in notebook 01, replicates all of that on every GPU. So the largest
model you can train is bounded by one GPU, no matter how many you have.

Sharding breaks that bound. Instead of replicating, you partition the
parameters, gradients, and optimizer states across the data-parallel GPUs. Each
GPU stores one slice and the framework gathers a full layer's weights only for
the moment it is computed, then frees them. The memory per GPU drops roughly in
proportion to the number of GPUs.

Two mature implementations do this. PyTorch FSDP, through the `fully_shard` API,
is native to PyTorch. DeepSpeed ZeRO, through `deepspeed.initialize`, offers
staged sharding. This notebook runs both on gpt2 across four T4 GPUs, and the
key thing to notice is that the Ray Train surface does not change. Same
`TorchTrainer`, same `ScalingConfig`, same `ray.train.report`.


## Key concepts used in this notebook

**Sharding.** Partition parameters, gradients, and optimizer states across the
data-parallel GPUs rather than replicating them. Gather a layer just in time for
compute, then free it.

**FSDP and `fully_shard`.** PyTorch's native sharding. You wrap each transformer
block and the root model with `fully_shard`, and PyTorch shards their parameters
across a device mesh.

**Mixed precision on T4.** We compute in fp16 because the T4 has no native bf16.
`MixedPrecisionPolicy` for FSDP and the fp16 block in the DeepSpeed config both
come from `common.utils`, which picks fp16 on this hardware.

**DeepSpeed ZeRO stages.** Stage 1 shards optimizer states, stage 2 adds
gradients, stage 3 adds parameters. Higher stage means less memory per GPU and
more communication.

**Sharded checkpoints.** When the model is split across GPUs, each GPU writes its
own slice. PyTorch Distributed Checkpoint and DeepSpeed's `save_checkpoint` both
do this, and Ray Train bundles the slices into one logical checkpoint.


## What you will learn

- Why replicated data parallel caps model size at one GPU
- How FSDP `fully_shard` splits a model across a device mesh
- How to confirm each rank holds only its slice of the parameters
- How DeepSpeed ZeRO stages trade memory for communication
- How sharded checkpoints are written by every rank and bundled by Ray Train
- That the Ray Train surface is identical whether you replicate or shard


## Why Ray on Anyscale for sharded training

| Challenge | Without Ray | With Ray |
|---|---|---|
| Launch sharded workers | Manual process groups and device mesh wiring | `TorchTrainer` plus `ScalingConfig` |
| FSDP and DeepSpeed | Different launchers and scripts | Same `train_loop_per_worker`, swap the body |
| Sharded checkpoints | Coordinate per-rank writes by hand | Each rank reports, Ray bundles to storage |
| Shared storage | Configure by hand | `/mnt/cluster_storage` on every node |
| Scale the shard group | Rewrite launch | Raise `num_workers` |


## Architecture

```
   Replicated data parallel (notebook 01)     Sharded (this notebook)
   ------------------------------------        ------------------------------
   GPU0  full model + optim (28 GB)            GPU0  shard 0 of model + optim
   GPU1  full model + optim (28 GB)            GPU1  shard 1 of model + optim
   GPU2  full model + optim (28 GB)            GPU2  shard 2 of model + optim
   GPU3  full model + optim (28 GB)            GPU3  shard 3 of model + optim
                                               gather a layer just in time,
                                               compute, then free it
```

Both run under the same Ray Train controller and workers. Sharding changes what
each worker holds, not how the job is launched.


## How this scales on Anyscale

| | This notebook | Production |
|---|---|---|
| Model | gpt2 (124M), shards are small | 7B to 70B and up, sharding is essential |
| GPUs | 4 T4 in one shard group | Many GPUs, often combined with tensor parallel (notebook 03) |
| Precision | fp16 on T4 | bf16 on Ampere and newer |
| Change needed | -- | Raise `num_workers`, pick a ZeRO stage or FSDP policy |

## Cell 1 — Connect and build the dataset

**What you do.** Connect to Ray and build a tokenized text dataset with the
helper from `common.utils`, which wraps the same Ray Data pattern you wrote by
hand in notebook 01.

**What to check.** Four GPUs are present and a tokenized batch has shape
(batch, 128).

**Why it matters.** Data loading is solved, so this notebook can focus on
sharding. Both training runs below feed from this one dataset.


In [1]:
import os
os.environ["RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO"] = "0"

import ray
from common import utils

if not ray.is_initialized():
    ray.init(address="auto", runtime_env=utils.build_runtime_env())

utils.print_cluster_resources()

MODEL_NAME = "gpt2"
train_ds = utils.build_tokenized_text_dataset(MODEL_NAME, seq_len=128)
print({k: v.shape for k, v in train_ds.take_batch(4).items()})


2026-07-16 08:33:35,339	INFO worker.py:1814 -- Connecting to existing Ray cluster at address: 10.0.135.10:6379...
2026-07-16 08:33:35,364	INFO worker.py:2003 -- Connected to Ray cluster. View the dashboard at https://session-ai1n2q9mz57t1ufynpn6k7e5xl.i.anyscaleuserdata.com 
2026-07-16 08:33:35,370	WARNING working_dir.py:86 -- Directory '.git' is now ignored by default when packaging the working directory. To disable this behavior, set the `RAY_OVERRIDE_RUNTIME_ENV_DEFAULT_EXCLUDES=''` environment variable.
2026-07-16 08:33:35,371	INFO packaging.py:392 -- Ignoring upload to cluster for these files: [PosixPath('/home/ray/default/ray_summit_foundation_model_training_2026/.gitignore')]
2026-07-16 08:33:35,376	INFO packaging.py:691 -- Creating a file package for local module '.'.
2026-07-16 08:33:35,377	INFO packaging.py:392 -- Ignoring upload to cluster for these files: [PosixPath('/home/ray/default/ray_summit_foundation_model_training_2026/.gitignore')]
2026-07-16 08:33:35,383	INFO packa

Ray cluster: 4 GPU(s), 16 CPU(s)
Accelerator type(s): L4


Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

(autoscaler +14s) Tip: use `ray status` to view detailed cluster status. To disable these messages, set RAY_SCHEDULER_EVENTS=0.


2026-07-16 08:34:52,522	INFO logging.py:416 -- Registered dataset logger for dataset dataset_2_0
2026-07-16 08:34:52,615	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_2_0. Full logs are in /tmp/ray/session_2026-07-16_07-09-33_113839_3150/logs/ray-data
2026-07-16 08:34:52,615	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_2_0: InputDataBuffer[Input] -> ActorPoolMapOperator[MapBatches(TokenizeText)] -> LimitOperator[limit=4]
{"asctime":"2026-07-16 08:34:52,646","levelname":"E","message":"Actor with class name: 'MapWorker(MapBatches(TokenizeText))' and ID: '8ccf8bbf9f015a7ffbd2700603000000' has constructor arguments in the object store and max_restarts > 0. If the arguments in the object store go out of scope or are lost, the actor restart will fail. See https://github.com/ray-project/ray/issues/53727 for more details.","filename":"core_worker.cc","lineno":2194}
2026-07-16 08:34:52,666	WARNING resource_manager.py:169 -- ⚠️  Ray's object store

{'input_ids': (4, 128), 'attention_mask': (4, 128)}


## Sharding with PyTorch FSDP

FSDP shards a model with the `fully_shard` API. You apply it to each repeated
unit, here the transformer blocks, and then to the root model. PyTorch splits
each unit's parameters across a device mesh built over the data-parallel GPUs.
During the forward and backward passes it gathers a unit's full weights just in
time, runs the computation, and frees them again.

We attach a `MixedPrecisionPolicy` so compute happens in fp16, the right choice
on a T4. The proof that sharding worked is simple. After `fully_shard`, each rank
should hold close to one quarter of the parameters on four GPUs.


## Cell 2 — The FSDP training loop

**What you do.** Define a loop that loads gpt2, shards it with `fully_shard`,
trains a few steps, and writes a distributed checkpoint. It also measures the
local parameter shard and the peak GPU memory.

**What to check.** Find `fully_shard` applied per block and to the root. The
checkpoint uses PyTorch Distributed Checkpoint, so every rank writes its own
slice and reports it.

**Why it matters.** This is the FSDP equivalent of notebook 01's loop. The only
real change from single-GPU code is the `fully_shard` calls and the distributed
checkpoint.


In [2]:
import ray.train
import ray.train.torch
from ray.train import Checkpoint

def fsdp_train_loop(config):
    import os, tempfile
    import torch
    from transformers import AutoModelForCausalLM
    from torch.distributed.fsdp import fully_shard, MixedPrecisionPolicy
    from torch.distributed.device_mesh import init_device_mesh
    import torch.distributed.checkpoint as dcp
    from torch.distributed.checkpoint.state_dict import get_state_dict
    from torch.distributed.checkpoint.stateful import Stateful
    import ray.train, ray.train.torch
    from ray.train import Checkpoint

    device = ray.train.torch.get_device()
    torch.cuda.set_device(device)
    world_size = ray.train.get_context().get_world_size()
    world_rank = ray.train.get_context().get_world_rank()

    model = AutoModelForCausalLM.from_pretrained(config["model_name"]).to(device)  # weights temporarily all on device -- if model is too big for even this, init the model on the meta device or CPU and load shards to GPU
    full_params = sum(p.numel() for p in model.parameters())

    # CHANGE 1: The single line prepare_model(model) is replaced by "build a mesh,
    # then call fully_shard on every repeated unit and the root." That's the whole trick.

    # Shard across a 1D device mesh over the data-parallel GPUs, computing in fp16.
    mesh = init_device_mesh("cuda", (world_size,), mesh_dim_names=("dp",))  # treat all GPUs as one sharding group
    mp_policy = MixedPrecisionPolicy(param_dtype=torch.float16, reduce_dtype=torch.float16)  # Store and communicate model weights in fp16
    for block in model.transformer.h:        # shard each transformer block across the GPUs (each GPU holds a slice of every block)
        fully_shard(block, mesh=mesh, mp_policy=mp_policy)
    fully_shard(model, mesh=mesh, mp_policy=mp_policy)   # shard the root (i.e the outer model container)

    # Proof of sharding. Sum the local slice of every parameter on this rank.
    local_params = sum(
        (p.to_local().numel() if hasattr(p, "to_local") else p.numel())
        for p in model.parameters()
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"])
    shard = ray.train.get_dataset_shard("train")
    per_worker_bs = config["global_batch_size"] // world_size

    torch.cuda.reset_peak_memory_stats()
    model.train()
    running, steps = 0.0, 0
    for batch in shard.iter_torch_batches(
        batch_size=per_worker_bs,
        dtypes={"input_ids": torch.long, "attention_mask": torch.long},
        device=device,
    ):
        ids = batch["input_ids"]
        loss = model(input_ids=ids, attention_mask=batch["attention_mask"], labels=ids).loss
        optimizer.zero_grad()
        
        # CHANGE 2:
        # In DDP, backward() triggers one all-reduce: every GPU already has full gradients, they just average them.
        #
        # In FSDP, backward() triggers an all-gather of each layer's parameter shards (to reconstruct the full layer for the math),
        # followed by a reduce-scatter of that layer's gradients (so each GPU keeps only its ¼ slice of the gradient), 
        # then frees the gathered weights. optimizer.step() then updates only the local shard.
        loss.backward()  # ← textually the SAME line as DDP which triggers an all reduce. 
        
        optimizer.step()
        running += loss.item()
        steps += 1
        if steps >= config["max_steps"]:
            break

    peak_gb = torch.cuda.max_memory_allocated() / 1e9

    # A sharded checkpoint. Every rank writes its own slice with PyTorch DCP.
    class AppState(Stateful):
        def __init__(self, model, optimizer):  # Store references to the real objects (do not copy the model nor optimizer -- just point at them) 
            self.model, self.optimizer = model, optimizer
        def state_dict(self):
            msd, osd = get_state_dict(self.model, self.optimizer)  # Collect the checkpoint state for this FSDP model and optimizer
            return {"model": msd, "optim": osd}
        def load_state_dict(self, sd):  # method tells PyTorch how to load the checkpoint back
            from torch.distributed.checkpoint.state_dict import set_state_dict
            set_state_dict(self.model, self.optimizer,
                           model_state_dict=sd["model"], optim_state_dict=sd["optim"])


    # CHANGE 3: no rank has the full model, so every rank writes its own slice using PyTorch Distributed Checkpoint (DCP)
    with tempfile.TemporaryDirectory() as ckpt_dir:
        dcp.save({"app": AppState(model, optimizer)}, checkpoint_id=ckpt_dir)  # Every worker/rank writes its own shard into the checkpoint directory. (calls AppState.state_dict implicitely) 
        checkpoint = Checkpoint.from_directory(ckpt_dir)  # Wrap the directory as a Ray Train checkpoint
        ray.train.report(
            {
                "loss": running / max(steps, 1),
                "full_params": full_params,
                "local_shard": local_params,
                "shard_fraction": round(local_params / full_params, 3),
                "peak_gb": round(peak_gb, 3),
            },
            checkpoint=checkpoint,
        )


(MapWorker(MapBatches(TokenizeText)) pid=10688, ip=10.0.137.173) [W716 08:34:59.094090336 AllocatorConfig.cpp:28] Warning: PYTORCH_CUDA_ALLOC_CONF is deprecated, use PYTORCH_ALLOC_CONF instead (function operator())


## Cell 3 — Launch FSDP

**What you do.** Run the FSDP loop on four GPUs with the same `TorchTrainer`
pattern from notebook 01.

**What to check.** `shard_fraction` is about 0.25, which means each of the four
GPUs holds a quarter of the parameters. The model was split, not replicated.

**Why it matters.** That one number is the whole point of sharding. On four GPUs
each rank holds a quarter of the model, and the fraction shrinks as you add GPUs.


In [3]:
from ray.train import ScalingConfig, RunConfig
from ray.train.torch import TorchTrainer

fsdp_trainer = TorchTrainer(
    train_loop_per_worker=fsdp_train_loop,
    train_loop_config={"model_name": MODEL_NAME, "lr": 5e-5,
                       "global_batch_size": 16, "max_steps": 20},
    scaling_config=ScalingConfig(num_workers=4, use_gpu=True),
    run_config=RunConfig(storage_path="/mnt/cluster_storage/ray_summit_sharding",
                         name="fsdp_gpt2"),
    datasets={"train": train_ds},
)
fsdp_result = fsdp_trainer.fit()
print("FSDP metrics:", fsdp_result.metrics)


(pid=gcs_server) {"asctime":"2026-07-16 08:35:00,814","levelname":"E","message":"Failed to kill actor 992aef01b8c6f6ce9d2e0ae803000000, return status: Invalid: KillActor RPC failed for actor 992aef01b8c6f6ce9d2e0ae803000000: RpcError: RPC error: Socket closed rpc_code: 14","component":"gcs_server","filename":"gcs_actor_scheduler.cc","lineno":499}
(TrainController pid=35507) A run snapshot was found in storage folder at: '/mnt/cluster_storage/ray_summit_sharding/fsdp_gpt2'
(TrainController pid=35507) This snapshot contains a list of checkpoints reported via `ray.train.report` and will be loaded. This allows the latest checkpoint found in the snapshot to be accessible within your training function via `ray.train.get_checkpoint`.
(TrainController pid=35507) If you meant to start a brand new training job without any information about previous checkpoints found in this directory, please configure a new, unique `RunConfig(name)` or delete the existing folder at '/mnt/cluster_storage/ray_summ

(pid=35843) Running Dataset train_3_0.: 0.00 row [00:00, ? row/s]

(pid=35843) - MapBatches(TokenizeText):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(pid=35843) - split(4, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(SplitCoordinator pid=35843) ⚠️  Ray's object store is configured to use only 27.1% of available memory (26.0GiB out of 96.0GiB total). For optimal Ray Data performance, we recommend setting the object store to at least 50% of available memory. You can do this by setting the 'object_store_memory' parameter when calling ray.init() or by setting the RAY_DEFAULT_OBJECT_STORE_MEMORY_PROPORTION environment variable.
(SplitCoordinator pid=35843) [dataset]: A new progress UI is available. To enable, set `ray.data.DataContext.get_current().enable_rich_progress_bars = True` and `ray.data.DataContext.get_current().use_ray_tqdm = False`.
(SplitCoordinator pid=35843) /home/ray/anaconda3/lib/python3.11/site-packages/ray/data/_internal/execution/operators/actor_pool_map_operator.py:471: UserWarning: The minimum number of concurrent actors for 'MapBatches(TokenizeText)' is set to 2, but the operator only received 1 input(s). This means that the operator can launch at most 1 task(s), and won't fully u

FSDP metrics: {'loss': 2.56240234375, 'full_params': 124439808, 'local_shard': 31110528, 'shard_fraction': 0.25, 'peak_gb': 1.08}


## Sharding with DeepSpeed ZeRO

DeepSpeed reaches the same goal through a config, not API calls on the model. You
hand your model and optimizer to `deepspeed.initialize` with a ZeRO stage, and it
returns an engine that shards state and drives the backward and optimizer steps.

The stages trade memory for communication.

| Stage | What it shards | Memory per GPU | Communication |
|---|---|---|---|
| ZeRO-1 | Optimizer states | Lower | Reduce-scatter plus all-gather |
| ZeRO-2 | Optimizer states and gradients | Lower still | Adds gradient reduce-scatter |
| ZeRO-3 | Optimizer, gradients, and parameters | Lowest, scales with GPUs | Adds parameter all-gather each layer |

ZeRO-3 is the closest analogue to FSDP, since both shard the parameters
themselves. We run stage 3 here. The fp16 block in the config comes from
`common.utils`, which selects fp16 for the T4.


## Cell 4 — The DeepSpeed training loop

**What you do.** Define a loop that wraps gpt2 with `deepspeed.initialize` at a
chosen ZeRO stage and trains. DeepSpeed drives the backward and step through its
engine.

**What to check.** The model goes in as plain PyTorch and comes back as a
DeepSpeed engine. You call `engine.backward(loss)` and `engine.step()` instead of
the optimizer directly. The checkpoint is DeepSpeed's own partitioned format.

**Why it matters.** Same training job, different sharding engine. The Ray Train
wrapper around it is unchanged from the FSDP loop.


In [4]:
def deepspeed_train_loop(config):
    import os, tempfile
    import torch, deepspeed
    from transformers import AutoModelForCausalLM
    import ray.train, ray.train.torch
    from ray.train import Checkpoint
    from common import utils

    world_size = ray.train.get_context().get_world_size()
    world_rank = ray.train.get_context().get_world_rank()

    # Change 1: wrap via a config + deepspeed.initialize (no mesh, no per-module calls)
    model = AutoModelForCausalLM.from_pretrained(config["model_name"])  # note: no .to(device)
    full_params = sum(p.numel() for p in model.parameters())

    deepspeed_config = {
        "train_micro_batch_size_per_gpu": config["global_batch_size"] // world_size,
        "zero_optimization": {"stage": config["stage"], "overlap_comm": True,
                              "contiguous_gradients": True},
        "gradient_clipping": 1.0,
    }
    deepspeed_config.update(utils.deepspeed_precision_config())   # fp16 

    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"])
    engine, _, _, _ = deepspeed.initialize(model=model, optimizer=optimizer, config=deepspeed_config)  # Initialize DeepSpeed engine
    device = engine.device  # DeepSpeed decides placement for you

    shard = ray.train.get_dataset_shard("train")
    torch.cuda.reset_peak_memory_stats()
    engine.train()
    running, steps = 0.0, 0
    for batch in shard.iter_torch_batches(
        batch_size=deepspeed_config["train_micro_batch_size_per_gpu"],
        dtypes={"input_ids": torch.long, "attention_mask": torch.long},
        device=device,
    ):
        ids = batch["input_ids"]
        # Change 2: the step is driven by the engine, not the optimizer
        loss = engine(input_ids=ids, attention_mask=batch["attention_mask"], labels=ids).loss
        engine.backward(loss)
        engine.step()
        running += loss.item()
        steps += 1
        if steps >= config["max_steps"]:
            break

    peak_gb = torch.cuda.max_memory_allocated() / 1e9

    # Change 3: checkpoint via the engine's own partitioned format Every rank participates.
    with tempfile.TemporaryDirectory() as ckpt_dir:
        engine.save_checkpoint(ckpt_dir, tag="step")
        checkpoint = Checkpoint.from_directory(ckpt_dir)
        ray.train.report(
            {"loss": running / max(steps, 1), "stage": config["stage"],
             "full_params": full_params, "peak_gb": round(peak_gb, 3)},
            checkpoint=checkpoint,
        )


(TrainController pid=35507) [State Transition] SHUTTING_DOWN -> FINISHED.


## Cell 5 — Launch DeepSpeed ZeRO-3

**What you do.** Run the DeepSpeed loop at ZeRO stage 3 on four GPUs.

**What to check.** It trains and reports a loss and peak memory, just like the
FSDP run. Switching `stage` to 1 or 2 changes the memory and communication
profile without touching the loop.

**Why it matters.** You now have two production-grade sharding engines behind the
same Ray Train surface. Pick the one your stack prefers.


In [5]:
deepspeed_trainer = TorchTrainer(
    train_loop_per_worker=deepspeed_train_loop,
    train_loop_config={"model_name": MODEL_NAME, "lr": 5e-5,
                       "global_batch_size": 16, "max_steps": 20, "stage": 3},
    scaling_config=ScalingConfig(num_workers=4, use_gpu=True),
    run_config=RunConfig(storage_path="/mnt/cluster_storage/ray_summit_sharding",
                         name="deepspeed_gpt2"),
    datasets={"train": train_ds},
)
deepspeed_result = deepspeed_trainer.fit()
print("DeepSpeed metrics:", deepspeed_result.metrics)


(TrainController pid=36153) A run snapshot was found in storage folder at: '/mnt/cluster_storage/ray_summit_sharding/deepspeed_gpt2'
(TrainController pid=36153) This snapshot contains a list of checkpoints reported via `ray.train.report` and will be loaded. This allows the latest checkpoint found in the snapshot to be accessible within your training function via `ray.train.get_checkpoint`.
(TrainController pid=36153) If you meant to start a brand new training job without any information about previous checkpoints found in this directory, please configure a new, unique `RunConfig(name)` or delete the existing folder at '/mnt/cluster_storage/ray_summit_sharding/deepspeed_gpt2'.
(TrainController pid=36153) Requesting resources: {'GPU': 1} * 4
(TrainController pid=36153) [State Transition] INITIALIZING -> SCHEDULING.
(TrainController pid=36153) Attempting to start training worker group of size 4 with the following resources: [{'GPU': 1}] * 4
(RayTrainWorker pid=10942, ip=10.0.189.137) Sett

(pid=36413) Running Dataset train_5_0.: 0.00 row [00:00, ? row/s]

(pid=36413) - MapBatches(TokenizeText):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(pid=36413) - split(4, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(SplitCoordinator pid=36413) {"asctime":"2026-07-16 08:36:05,021","levelname":"E","message":"Actor with class name: 'MapWorker(MapBatches(TokenizeText))' and ID: '40f3d3980a86c3c63822e1ae03000000' has constructor arguments in the object store and max_restarts > 0. If the arguments in the object store go out of scope or are lost, the actor restart will fail. See https://github.com/ray-project/ray/issues/53727 for more details.","filename":"core_worker.cc","lineno":2194}
(SplitCoordinator pid=36413) ⚠️  Ray's object store is configured to use only 27.1% of available memory (26.0GiB out of 96.0GiB total). For optimal Ray Data performance, we recommend setting the object store to at least 50% of available memory. You can do this by setting the 'object_store_memory' parameter when calling ray.init() or by setting the RAY_DEFAULT_OBJECT_STORE_MEMORY_PROPORTION environment variable.
(SplitCoordinator pid=36413) [dataset]: A new progress UI is available. To enable, set `ray.data.DataContext.ge

DeepSpeed metrics: {'loss': 2.68359375, 'stage': 3, 'full_params': 124439808, 'peak_gb': 2.235}


(TrainController pid=36153) [State Transition] SHUTTING_DOWN -> FINISHED.


## FSDP or DeepSpeed, which one

Both shard the same state and reach similar memory. The choice is usually about
the ecosystem you already use.

| | FSDP | DeepSpeed ZeRO |
|---|---|---|
| Origin | Native PyTorch | Microsoft library |
| How you enable it | `fully_shard` calls on modules | A config passed to `initialize` |
| Granularity | Per-module, very flexible | Per-stage, simple to switch |
| Composes with tensor parallel | Built on DTensor, composes cleanly | AutoTP, with ZeRO 1 and 2 |
| Best when | You want PyTorch-native control | You want config-driven stages and offload |

Notebook 03 builds directly on this. FSDP composes with tensor parallelism to
form 2D parallelism, because both are built on the same DTensor foundation.


## Conclusion

You sharded gpt2 two ways and watched each rank hold only a quarter of the
parameters on four GPUs. FSDP did it with `fully_shard` on the modules. DeepSpeed
did it with a ZeRO stage in a config. Both wrote sharded checkpoints that Ray
Train bundled to shared storage, and both ran under the same `TorchTrainer` you
have used since notebook 01.

Ray and PyTorch primitives you used. `fully_shard`, `MixedPrecisionPolicy`,
`init_device_mesh`, PyTorch Distributed Checkpoint, `deepspeed.initialize` with
ZeRO, and the unchanged `TorchTrainer`, `get_dataset_shard`, and
`ray.train.report`.

Sharding solves memory along the data-parallel axis. But some models are so large
that even a single layer's compute is too much for one GPU, or tensor
communication dominates. In notebook 03 you add a second axis, tensor
parallelism, and combine it with data parallelism for 2D parallelism.
